# Sutton & Barto Chapter 3: Small Gridworld Case Study
## A Finite MDP Worked Example

This notebook ties Chapter 3 ideas together with a very small deterministic gridworld-style MDP.

## What You Will See

- a tiny 2x2 gridworld finite MDP
- deterministic transitions with one terminal goal
- iterative evaluation of a random policy
- greedy improvement from state values

Dependencies: `numpy`, `matplotlib`

## Environment setup

We use four states arranged in a 2x2 grid:

- `0`: top-left
- `1`: top-right
- `2`: bottom-left
- `3`: bottom-right terminal goal

Every non-terminal move gives reward `-1`, and entering the terminal state gives reward `0`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')

In [ ]:
gamma = 0.9
states = [0, 1, 2, 3]
terminal_state = 3
actions = ['up', 'down', 'left', 'right']

coords = {0: (0, 0), 1: (0, 1), 2: (1, 0), 3: (1, 1)}
state_from_coord = {v: k for k, v in coords.items()}

def step(state, action):
    if state == terminal_state:
        return terminal_state, 0.0

    r, c = coords[state]
    deltas = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}
    dr, dc = deltas[action]
    nr = min(1, max(0, r + dr))
    nc = min(1, max(0, c + dc))
    next_state = state_from_coord[(nr, nc)]
    reward = 0.0 if next_state == terminal_state else -1.0
    return next_state, reward

In [ ]:
policy = {s: {a: 0.25 for a in actions} for s in states if s != terminal_state}

def evaluate_policy(iterations=50):
    v = {s: 0.0 for s in states}
    history = []
    for _ in range(iterations):
        new_v = v.copy()
        for s in states:
            if s == terminal_state:
                continue
            total = 0.0
            for a, prob in policy[s].items():
                next_state, reward = step(s, a)
                total += prob * (reward + gamma * v[next_state])
            new_v[s] = total
        v = new_v
        history.append([v[s] for s in states])
    return v, np.array(history)


v_pi, history = evaluate_policy()
v_pi

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for s in [0, 1, 2]:
    ax.plot(history[:, s], label=f'state {s}')
ax.set_title('Convergence of State Values Under a Random Policy')
ax.set_xlabel('Iteration')
ax.set_ylabel('Value estimate')
ax.legend()
plt.tight_layout()

## Value table snapshot

We can also reshape the final state values back into the 2x2 grid. This gives a more spatial view of the same information.

In [ ]:
value_grid = np.array([[v_pi[0], v_pi[1]], [v_pi[2], v_pi[3]]])

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(value_grid, cmap='viridis')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{value_grid[i, j]:.2f}', ha='center', va='center', color='white')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_title('State Values in Grid Form')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()

## Greedy improvement

Once we have $v_\pi$, we can do one-step lookahead and choose the action with the highest expected return.

In [ ]:
greedy_policy = {}
for s in [0, 1, 2]:
    action_values = {}
    for a in actions:
        next_state, reward = step(s, a)
        action_values[a] = reward + gamma * v_pi[next_state]
    greedy_policy[s] = max(action_values, key=action_values.get)

greedy_policy

## Takeaway

- A tiny gridworld is enough to see Chapter 3 ideas working concretely.
- Policy evaluation estimates how good each state is under the current policy.
- Greedy improvement converts those values into better actions.
- Visualizing the state values makes the geometry of the task easier to interpret.
- This is the conceptual bridge into dynamic programming chapters.

## Reference

Richard S. Sutton and Andrew G. Barto, *Reinforcement Learning: An Introduction*, Chapter 3.